# Week 3 - Ames Housing: Cleaning and Outlier Detection

This notebook covers the **Cleaning Sprint** for the Ames Housing dataset. The main focus this week is to understand the missing values, decide how to handle them, and look for unusual observations using boxplots and Z-scores.

The goal is not to make the data look perfect. It is to make sensible cleaning decisions and keep a record of why those decisions were made.

## 1. Import the libraries and load the data

The project data may be kept as a CSV file. The code below checks for `AmesHousing.csv` first. If that file is not present, it also accepts the original Excel file so the notebook can still be run without changing the rest of the code.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

if os.path.exists("AmesHousing.csv"):
    df = pd.read_csv("AmesHousing.csv")
    print("Loaded AmesHousing.csv")
elif os.path.exists("AmesHousing.xlsx"):
    df = pd.read_excel("AmesHousing.xlsx")
    print("Loaded AmesHousing.xlsx")
else:
    raise FileNotFoundError("Place AmesHousing.csv (or AmesHousing.xlsx) in the same folder as this notebook.")

print("Shape:", df.shape)

## 2. First look at the dataset

Before changing anything, it is useful to see the first few rows, the column types, and the size of the dataset. This gives us a reference point for the cleaning work.

In [ ]:
display(df.head())

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

## 3. Check for missing values

The first cleaning task is to find where values are missing. Looking at both the count and the percentage is more useful than looking at the count alone because the columns all have the same number of rows.

In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().sum() / len(df) * 100).round(2)
})

missing = missing[missing["Missing Values"] > 0].sort_values(
    "Missing Values", ascending=False
)

display(missing)

## 4. Decide what the missing values mean

A missing value does not always mean that someone forgot to enter the data. In this dataset, several missing categorical values actually mean that the house does not have that feature. For example, a missing `Pool QC` means there is no pool, not that the pool quality was forgotten.

For those columns, replacing the missing value with `None` keeps that information instead of throwing it away.

For numerical columns, the strategy depends on the variable. For example, `Lot Frontage` can reasonably use its median because it is a continuous measurement and housing data can be skewed by unusually large properties.

In [ ]:
# Missing categorical values that represent the absence of a feature
no_feature_cols = [
    "Alley", "Mas Vnr Type", "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
    "BsmtFin Type 1", "BsmtFin Type 2", "Fireplace Qu", "Garage Type",
    "Garage Finish", "Garage Qual", "Garage Cond", "Pool QC", "Fence",
    "Misc Feature"
]

for col in no_feature_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")

# Numerical measurements where a missing value means the feature is absent
zero_if_absent = [
    "Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF",
    "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath", "Garage Cars",
    "Garage Area"
]

for col in zero_if_absent:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Lot Frontage is a numerical measurement, so use the median.
if "Lot Frontage" in df.columns:
    df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

# If Garage Year is missing because there is no garage, use 0 as a clear marker.
if "Garage Yr Blt" in df.columns:
    df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(0)

print("Main missing-value replacements have been applied.")

## 5. Check what is still missing

After the first cleaning pass, we check the dataset again. Any remaining missing values should be small enough to investigate rather than filling everything automatically.

In [ ]:
remaining_missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().sum() / len(df) * 100).round(2)
})

remaining_missing = remaining_missing[
    remaining_missing["Missing Values"] > 0
].sort_values("Missing Values", ascending=False)

display(remaining_missing)

If a few numerical values are still missing, using the median is a reasonable final step for this exploratory project. The median is less affected by extreme house prices and unusually large measurements than the mean.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    if df[col].isna().any():
        mode = df[col].mode()
        if not mode.empty:
            df[col] = df[col].fillna(mode.iloc[0])

print("Remaining missing values:", int(df.isna().sum().sum()))

## 6. Outlier detection with boxplots

The next step is to look for unusually high or low values. Boxplots are useful because they show the middle 50% of the data and make extreme observations easy to spot.

An outlier is not automatically a mistake. A large house or an expensive house can be a genuine observation, so these plots are mainly used to identify records that deserve a closer look.

In [ ]:
boxplot_cols = [
    "SalePrice", "Gr Liv Area", "Total Bsmt SF",
    "1st Flr SF", "Garage Area", "Overall Qual"
]

for col in boxplot_cols:
    if col in df.columns:
        plt.figure(figsize=(9, 4))
        sns.boxplot(x=df[col])
        plt.title(f"Boxplot of {col}")
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()

## 7. Z-score check

A Z-score tells us how far a value is from the average in terms of standard deviations. For this project, values with an absolute Z-score greater than 3 will be flagged as potential outliers.

This is only a screening method. We are not deleting every row that crosses the threshold.

In [ ]:
zscore_cols = [
    "SalePrice", "Gr Liv Area", "Total Bsmt SF",
    "1st Flr SF", "Garage Area"
]

zscore_summary = []

for col in zscore_cols:
    if col in df.columns:
        values = df[col].astype(float)
        z = (values - values.mean()) / values.std(ddof=0)
        count = int((z.abs() > 3).sum())
        zscore_summary.append({
            "Column": col,
            "Potential Outliers (|Z| > 3)": count
        })

zscore_summary = pd.DataFrame(zscore_summary)
display(zscore_summary)

## 8. Inspect some potential outliers

Rather than removing the flagged rows immediately, we can inspect the most extreme observations. This helps separate unusual but believable houses from values that look like possible data-entry problems.

In [ ]:
if "SalePrice" in df.columns and "Gr Liv Area" in df.columns:
    sale_z = (df["SalePrice"] - df["SalePrice"].mean()) / df["SalePrice"].std(ddof=0)
    area_z = (df["Gr Liv Area"] - df["Gr Liv Area"].mean()) / df["Gr Liv Area"].std(ddof=0)

    possible_outliers = df.loc[
        (sale_z.abs() > 3) | (area_z.abs() > 3),
        ["Order", "SalePrice", "Gr Liv Area", "Overall Qual", "Year Built"]
    ].copy()

    display(possible_outliers.sort_values("SalePrice", ascending=False).head(15))

## 9. Cleaning decision

For this stage, the potential outliers are kept in the dataset unless there is clear evidence that a value is incorrect. The purpose of Week 3 is to identify and understand unusual observations, not to remove data just because it is extreme.

This also gives us a cleaner starting point for the Week 4 EDA, where the distributions and relationships between variables will be explored in more detail.

In [ ]:
print("Final dataset shape:", df.shape)
print("Total remaining missing values:", int(df.isna().sum().sum()))

display(df.head())

## 10. Save the cleaned dataset

The cleaned data is saved as a new CSV so the original dataset stays untouched. This cleaned file can be used as the starting point for the next stage of the project.

In [ ]:
output_file = "AmesHousing_Cleaned.csv"
df.to_csv(output_file, index=False)

print(f"Saved cleaned dataset as: {output_file}")